# Feature Engineering — Phase 4

**Baseline:** Ridge Regression — RMSE Test: **2.3691 mpg** | R²: **0.8956**

| Eksperimen | Perubahan |
|---|---|
| Baseline | Ridge tanpa perubahan |
| Exp A | Drop `kapasitas_mesin` (multikolinear dengan `kekuatan_mesin`) |
| Exp B | Tambah `power_to_weight` = `kekuatan_mesin` / `berat_mobil` |
| Exp C | Ubah `tahun_rilis` → `umur_mobil` = 82 − tahun (numerik kontinu) |
| Exp D | Gabungan A + B + C |

---
## 1. Setup & Load Data

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline  import Pipeline
from sklearn.compose   import TransformedTargetRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics   import mean_squared_error, r2_score

from src.data.load_data        import load_raw, split_features_target, split_train_test
from src.features.build_features import (
    NUM_COLS, ORD_SILINDER, ORD_TAHUN, OHE_COLS,
    MAX_TAHUN, get_tahun_cats,
    PowerToWeightAdder, CarAgeAdder, FullFeatureEngineer,
    build_preprocessor, build_preprocessor_no_tahun, build_target_transformer
)
from src.models.train        import evaluate, save_model
from src.visualization.plots import (
    save_fig, plot_actual_vs_pred, plot_fe_comparison
)

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
COLORS = sns.color_palette('muted')
print('Setup selesai.')

In [ ]:
df_raw = load_raw()
get_tahun_cats(df_raw)

X, y = split_features_target(df_raw)
X_train, X_test, y_train, y_test = split_train_test(X, y)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
all_results = []

def run_exp(name, pipeline):
    """Jalankan satu eksperimen, simpan hasil, return (pipeline, y_pred, result)."""
    result = evaluate(name, pipeline, X_train, y_train, X_test, y_test)
    y_pred = result.pop('_y_pred')
    pipe   = result.pop('_pipeline')
    print(f"  RMSE Test={result['RMSE Test']:.4f} | R²={result['R² Test']:.4f}")
    all_results.append(result)
    return pipe, y_pred, result


def ridge_pipeline(preprocessor=None, feature_step=None):
    """Helper: bangun pipeline Ridge dengan optional feature step di awal."""
    steps = []
    if feature_step:
        steps.append(feature_step)
    steps.append(('preprocessor', preprocessor or build_preprocessor()))
    steps.append(('model', TransformedTargetRegressor(
        regressor   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5),
        transformer = build_target_transformer()
    )))
    return Pipeline(steps)

---
## 2. Baseline — Ridge Regression

In [ ]:
print('── Baseline ──')
baseline_pipe, baseline_pred, baseline_res = run_exp('Baseline (Ridge)', ridge_pipeline())

---
## 3. Exp A — Drop `kapasitas_mesin`
`kapasitas_mesin` berkorelasi tinggi dengan `kekuatan_mesin` → multikolinearitas.

In [ ]:
# Konfirmasi korelasi
r = df_raw[['kapasitas_mesin', 'kekuatan_mesin']].corr().iloc[0, 1]
print(f'Korelasi kapasitas_mesin vs kekuatan_mesin: {r:.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df_raw[NUM_COLS].corr(), annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title('Korelasi Fitur Numerik', fontweight='bold')
plt.tight_layout()
save_fig('fe_exp_a_korelasi')
plt.show()

In [ ]:
NUM_A = ['kekuatan_mesin', 'berat_mobil', 'akselerasi']  # tanpa kapasitas_mesin
print('── Exp A: Drop kapasitas_mesin ──')
exp_a_pipe, exp_a_pred, exp_a_res = run_exp(
    'Exp A: Drop kapasitas_mesin',
    ridge_pipeline(preprocessor=build_preprocessor(num_cols=NUM_A))
)
delta_a = exp_a_res['RMSE Test'] - baseline_res['RMSE Test']
print(f'Δ RMSE vs Baseline: {delta_a:+.4f} → {"✅ Lebih baik" if delta_a < 0 else "❌ Lebih buruk"}')

---
## 4. Exp B — Tambah `power_to_weight`
Rasio `kekuatan_mesin / berat_mobil` menangkap interaksi tenaga vs berat secara langsung.

In [ ]:
NUM_B = NUM_COLS + ['power_to_weight']
print('── Exp B: + power_to_weight ──')
exp_b_pipe, exp_b_pred, exp_b_res = run_exp(
    'Exp B: + power_to_weight',
    ridge_pipeline(
        preprocessor=build_preprocessor(num_cols=NUM_B),
        feature_step=('feature_adder', PowerToWeightAdder())
    )
)
delta_b = exp_b_res['RMSE Test'] - baseline_res['RMSE Test']
print(f'Δ RMSE vs Baseline: {delta_b:+.4f} → {"✅ Lebih baik" if delta_b < 0 else "❌ Lebih buruk"}')

---
## 5. Exp C — Ubah `tahun_rilis` → `umur_mobil`
`umur_mobil = 82 − tahun_rilis` sebagai fitur numerik kontinu.

In [ ]:
NUM_C = NUM_COLS + ['umur_mobil']
print('── Exp C: tahun_rilis → umur_mobil ──')
exp_c_pipe, exp_c_pred, exp_c_res = run_exp(
    'Exp C: tahun → umur_mobil',
    ridge_pipeline(
        preprocessor=build_preprocessor_no_tahun(num_cols_with_age=NUM_C),
        feature_step=('age_adder', CarAgeAdder(max_tahun=MAX_TAHUN))
    )
)
delta_c = exp_c_res['RMSE Test'] - baseline_res['RMSE Test']
print(f'Δ RMSE vs Baseline: {delta_c:+.4f} → {"✅ Lebih baik" if delta_c < 0 else "❌ Lebih buruk"}')

---
## 6. Exp D — Kombinasi A + B + C

In [ ]:
NUM_D = ['kekuatan_mesin', 'berat_mobil', 'akselerasi', 'power_to_weight', 'umur_mobil']
print('── Exp D: Kombinasi A+B+C ──')
exp_d_pipe, exp_d_pred, exp_d_res = run_exp(
    'Exp D: Kombinasi (A+B+C)',
    ridge_pipeline(
        preprocessor=build_preprocessor_no_tahun(num_cols_with_age=NUM_D),
        feature_step=('fe', FullFeatureEngineer(max_tahun=MAX_TAHUN))
    )
)
delta_d = exp_d_res['RMSE Test'] - baseline_res['RMSE Test']
print(f'Δ RMSE vs Baseline: {delta_d:+.4f} → {"✅ Lebih baik" if delta_d < 0 else "❌ Lebih buruk"}')

---
## 7. Perbandingan Semua Eksperimen

In [ ]:
results_df = pd.DataFrame(all_results).set_index('Eksperimen')
baseline_rmse = results_df.loc['Baseline (Ridge)', 'RMSE Test']
results_df['Δ RMSE Test'] = (results_df['RMSE Test'] - baseline_rmse).round(4)
results_df['Δ R² Test']   = (results_df['R² Test'] - results_df.loc['Baseline (Ridge)', 'R² Test']).round(4)

display(results_df)
best_exp = results_df['RMSE Test'].idxmin()
print(f'\n🏆 Eksperimen terbaik: {best_exp}')
print(f'   RMSE Test: {results_df.loc[best_exp, "RMSE Test"]:.4f} mpg')
print(f'   R² Test  : {results_df.loc[best_exp, "R² Test"]:.4f}')
print(f'   Δ RMSE   : {results_df.loc[best_exp, "Δ RMSE Test"]:+.4f} mpg')

In [ ]:
plot_fe_comparison(results_df, baseline_key='Baseline (Ridge)', figname='fe_comparison_all')

In [ ]:
preds = {
    'Baseline': baseline_pred, 'Exp A': exp_a_pred,
    'Exp B': exp_b_pred, 'Exp C': exp_c_pred, 'Exp D': exp_d_pred
}
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for i, (name, pred) in enumerate(preds.items()):
    plot_actual_vs_pred(y_test.values, pred, name, color_idx=i, ax=axes[i], show=False)
fig.suptitle('Aktual vs Prediksi — Semua Eksperimen', fontsize=12, fontweight='bold')
plt.tight_layout()
save_fig('fe_actual_vs_pred_all')
plt.show()

---
## 8. Save Model Final

In [ ]:
import joblib

# Model terbaik = Exp A (Drop kapasitas_mesin + Ridge)
model_path = save_model(exp_a_pipe, 'ridge_final.pkl')

print(f'\nSpesifikasi model final:')
print(f'  Algoritma   : Ridge Regression (RidgeCV)')
print(f'  Alpha       : {exp_a_pipe.named_steps["model"].regressor_.alpha_:.4f}')
print(f'  Fitur input : {NUM_A + ORD_SILINDER + ORD_TAHUN + OHE_COLS}')
print(f'  RMSE Test   : {exp_a_res["RMSE Test"]:.4f} mpg')
print(f'  R² Test     : {exp_a_res["R² Test"]:.4f}')

In [ ]:
# Verifikasi load
loaded = joblib.load(model_path)
y_pred_verify = loaded.predict(X_test)
rmse_verify = np.sqrt(mean_squared_error(y_test, y_pred_verify))
assert abs(rmse_verify - exp_a_res['RMSE Test']) < 1e-4, 'Prediksi tidak cocok!'
print(f'Model terverifikasi. RMSE = {rmse_verify:.4f} mpg')

# Contoh prediksi
print('\nContoh prediksi 3 sampel:')
for i, (actual, pred) in enumerate(zip(y_test.values[:3], loaded.predict(X_test.iloc[:3]))):
    print(f'  [{i+1}] Aktual={actual:.2f} | Prediksi={pred:.2f} | Error={abs(actual-pred):.2f} mpg')